# Ce que vous saurez fabriquer en trois clics · *What you'll be able to make in three clicks*

Notebook compagnon du chapitre **18. Atelier données : découvrir FRED, l'entrepôt de la Réserve fédérale** — [lire l'article](https://nmlab.io/ressources/atelier-donnees-decouvrir-fred).
Companion notebook to chapter **18. Data Workshop: Discovering FRED** — [read the article](https://nmlab.io/en/ressources/data-workshop-discovering-fred).

**Exécutez tout** (*Exécution ▸ Tout exécuter* — *Runtime ▸ Run all*) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` dans la première cellule pour les libellés anglais. — Run all to rebuild the figure with **today's FRED data**; set `LANG = "en"` for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "en" → libellés anglais / English labels

# Style NMLab (thème sombre + police Inter) / NMLab style (dark theme + Inter font)
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm
nm.setup()

In [ ]:
import pandas as pd

# Données FRED en direct (CSV public, sans clé API) / live FRED data (no API key)
FRED = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="
unrate = pd.read_csv(FRED + "UNRATE", index_col="observation_date", parse_dates=True)["UNRATE"]
usrec  = pd.read_csv(FRED + "USREC",  index_col="observation_date", parse_dates=True)["USREC"]
usrec  = usrec.loc[unrate.index.min():]   # même fenêtre que le chômage / same window
unrate.tail()

In [ ]:
L = {
    "fr": dict(
        title="Ce que vous saurez fabriquer en trois clics",
        sub="Taux de chômage américain, avec les récessions du NBER — un graphique FRED type",
        ylab="taux de chômage, %",
        bands="bandes grises =\nrécessions (NBER)",
        note="Une série, une case « bandes de récession » à cocher, et l'histoire du cycle apparaît. Chaque pic de chômage\n"
             "épouse une bande grise. Source : BLS et NBER via FRED (UNRATE, USREC)."),
    "en": dict(
        title="What you'll be able to make in three clicks",
        sub="U.S. unemployment rate, with NBER recessions — a typical FRED chart",
        ylab="unemployment rate, %",
        bands="grey bands =\nrecessions (NBER)",
        note="One series, one « recession bars » box to tick, and the history of the cycle appears. Every unemployment\n"
             "peak hugs a grey band. Source: BLS and NBER via FRED (UNRATE, USREC)."),
}[LANG]

import matplotlib.dates as mdates

fig = nm.figure(height_px=1045)
ax = nm.axes(fig)

# Chaque période contiguë où USREC == 1 devient une bande grise
# / each contiguous run of USREC == 1 becomes a grey band
runs = usrec.ne(usrec.shift()).cumsum()
for _, seg in usrec.groupby(runs):
    if seg.iloc[0] == 1:
        ax.axvspan(seg.index[0], seg.index[-1], color=nm.COLORS["edge"],
                   alpha=0.75, linewidth=0)

ax.plot(unrate.index, unrate, color=nm.COLORS["blue"], linewidth=2.9)
ax.set_ylim(0, 15.5)
ax.set_yticks(range(0, 15, 2))
ax.set_ylabel(L["ylab"])
ax.margins(x=0.012)
ax.xaxis.set_major_locator(mdates.YearLocator(10))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.text(0.085, 0.70, L["bands"], transform=ax.transAxes, fontsize=21.5,
        color=nm.COLORS["muted"], linespacing=1.55)
nm.header(fig, L["title"], L["sub"])
nm.footer(fig, L["note"])